# 07 · Double Machine Learning
A check on the DiD result that does not assume prices react *linearly* to gas, weather and renewables (Section 2.2.4 of the thesis). Two machine-learning models are trained:
1. predict the **price** from the controls,
2. predict the **data center exposure** from the same controls.

Then we regress what is left of the price on what is left of the exposure; the slope θ is the data center effect (Chernozhukov et al., 2018).

**Fixed effects first.** Exposure is almost a pure function of zone and time, so a flexible model could predict it almost perfectly and leave nothing to learn from. As in the DiD, we first remove zone and month averages from every variable (the panel is balanced, so this is exact).

In [1]:
import numpy as np
import pandas as pd
import doubleml as dml
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from utils import CLEAN

zone = pd.read_parquet(CLEAN / "panel_zone_month.parquet")
X = ["gas_real", "hdd", "cdd", "renew_share", "rggi_real"]
Y_COLS = ["wholesale_real", "lmp_real", "capacity_mwh_real"]

def demean(df, cols):
    """Remove zone and month averages: x - zone mean - month mean + overall mean."""
    out = df[cols] - df.groupby("zone")[cols].transform("mean") - df.groupby("month")[cols].transform("mean") + df[cols].mean()
    return out

data = demean(zone, Y_COLS + ["dc_share"] + X)

## Two learners: random forest and gradient boosting
Five-fold cross-fitting; the random seeds make the results reproducible.

In [2]:
LEARNERS = {"Random forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=1, n_jobs=-1),
            "Gradient boosting": GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=1)}

np.random.seed(42)                       # fixes the random sample split, so results are reproducible
rows = []
for y in Y_COLS:
    for name, learner in LEARNERS.items():
        d = dml.DoubleMLData(data, y_col=y, d_cols="dc_share", x_cols=X)
        m = dml.DoubleMLPLR(d, ml_l=learner, ml_m=learner, n_folds=5)
        m.fit()
        lo, hi = m.confint().iloc[0]
        rows.append({"outcome": y, "learner": name, "theta": m.coef[0], "se": m.se[0], "ci_low": lo, "ci_high": hi})
dml_res = pd.DataFrame(rows)

## In $/MWh for Dominion
Same exposure change as in notebook 05: 30.6 $/MWh (random forest) and 26.7 (gradient boosting), Table B.3 of the thesis.

In [3]:
dom = zone[zone.zone == "DOM"]
d_exp = dom[dom.month >= "2025-08-01"].dc_share.mean() - dom[dom.month <= "2021-12-01"].dc_share.mean()
for c in ["theta", "ci_low", "ci_high"]:
    dml_res[c + "_dom"] = dml_res[c] * d_exp
dml_res.to_csv(CLEAN / "dml_results.csv", index=False)
dml_res.round(2)

,outcome,learner,theta,se,ci_low,ci_high,theta_dom,ci_low_dom,ci_high_dom
0,wholesale_real,Random forest,64.79,11.56,42.14,87.43,30.62,19.91,41.32
1,wholesale_real,Gradient boosting,56.51,10.71,35.51,77.51,26.71,16.78,36.63
2,lmp_real,Random forest,51.33,10.63,30.48,72.17,24.25,14.41,34.10
3,lmp_real,Gradient boosting,41.96,11.55,19.31,64.61,19.83,9.13,30.53
4,capacity_mwh_real,Random forest,17.39,4.95,7.68,27.09,8.22,3.63,12.80
5,capacity_mwh_real,Gradient boosting,14.89,4.05,6.94,22.84,7.04,3.28,10.79
